# Reservoir Shape Explorer

An interactive look at how Lake Powell and Lake Mead's footprint and depth change
over time, driven by a time slider.

Each reservoir gets two schematic subplots at the slider's selected date:

- **Perimeter** — a circle whose area matches the reservoir's real surface area at
  that date's elevation (from Reclamation's official area-capacity tables), viewed
  from above.
- **Floor profile** — a cross-section built from the same tables: the valley width
  at each elevation is set so its area matches the real surface area there, and the
  current water elevation is filled in blue.

These are **schematic, not surveyed shapes** — a circle stands in for the actual,
irregular canyon shoreline. What's real is the area-vs-elevation relationship
driving how fast that circle grows or shrinks; the true bathymetry would require
multi-gigabyte GIS terrain rasters this notebook doesn't attempt to load. See
[`src/colorado_river_viz/reservoir_geometry.py`](../src/colorado_river_viz/reservoir_geometry.py)
for the method and [`data/reservoir_area_capacity/`](../data/reservoir_area_capacity/)
for the underlying tables and their sources.


In [ ]:
from datetime import date, timedelta

import matplotlib.pyplot as plt
import requests
from matplotlib.widgets import Slider

from colorado_river_viz import (
    LAKE_MEAD,
    LAKE_POWELL,
    DateRange,
    fetch_rise_time_series,
    load_area_capacity_table,
    schematic_floor_profile_miles,
    schematic_perimeter_miles,
)

%matplotlib widget

years = 100
DATE_RANGE = DateRange(
    start=date.today() - timedelta(days=years * 365), end=date.today()
)

try:
    powell_elevation = fetch_rise_time_series(
        catalog_item_id=LAKE_POWELL.elevation_catalog_item_id, date_range=DATE_RANGE
    )
    mead_elevation = fetch_rise_time_series(
        catalog_item_id=LAKE_MEAD.elevation_catalog_item_id, date_range=DATE_RANGE
    )
except requests.exceptions.RequestException as exc:
    print(f"RISE API request failed ({exc}); cannot build the shape explorer.")
    powell_elevation = None
    mead_elevation = None

In [ ]:
if powell_elevation is not None and mead_elevation is not None:
    # Monthly resolution keeps the slider responsive to drag over a multi-decade
    # range; daily data makes the widget sluggish and near-impossible to land on
    # a specific date with the mouse.
    powell_monthly = powell_elevation["result"].resample("MS").mean()
    mead_monthly = mead_elevation["result"].resample("MS").mean()

    common_dates = powell_monthly.index.intersection(mead_monthly.index).sort_values()
    powell_monthly = powell_monthly.loc[common_dates]
    mead_monthly = mead_monthly.loc[common_dates]

    powell_table = load_area_capacity_table(LAKE_POWELL)
    mead_table = load_area_capacity_table(LAKE_MEAD)

    start, end = common_dates[0].date(), common_dates[-1].date()
    print(f"{len(common_dates)} months, {start} to {end}")

In [ ]:
if powell_elevation is not None and mead_elevation is not None:
    import numpy as np

    WATER_COLOR = "#4393C3"
    ROCK_COLOR = "#C2B280"
    RESERVOIRS = [
        (LAKE_POWELL, powell_table, powell_monthly, "#0072B2"),
        (LAKE_MEAD, mead_table, mead_monthly, "#E69F00"),
    ]
    # Fixed axis limits (set once, from full-pool size) keep the frame steady as
    # the slider moves, so shrinkage reads as the reservoir shrinking, not the
    # axes rescaling.
    max_radius_miles = {}
    for reservoir, table, _, _ in RESERVOIRS:
        full_pool_elev = reservoir.full_pool_elevation_ft
        full_pool_x, _ = schematic_perimeter_miles(table, full_pool_elev)
        max_radius_miles[reservoir.name] = full_pool_x.max()

    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    plt.subplots_adjust(bottom=0.17, hspace=0.4, wspace=0.3)
    slider_ax = fig.add_axes((0.15, 0.04, 0.7, 0.03))
    slider = Slider(
        slider_ax,
        "Month",
        0,
        len(common_dates) - 1,
        valinit=len(common_dates) - 1,
        valstep=1,
    )
    slider.valtext.set_text(common_dates[-1].strftime("%Y-%m"))

    def draw(_val: float) -> None:
        idx = int(slider.val)
        current_date = common_dates[idx]
        slider.valtext.set_text(current_date.strftime("%Y-%m"))

        for row, (reservoir, table, monthly_elevation, color) in enumerate(RESERVOIRS):
            elevation_ft = float(monthly_elevation.iloc[idx])
            map_ax, profile_ax = axes[row]

            map_ax.clear()
            x, y = schematic_perimeter_miles(table, elevation_ft)
            map_ax.fill(x, y, color=color, alpha=0.7)
            pad = max_radius_miles[reservoir.name] * 1.15
            map_ax.set_xlim(-pad, pad)
            map_ax.set_ylim(-pad, pad)
            map_ax.set_aspect("equal")
            map_ax.set_title(f"{reservoir.name} perimeter (schematic)")
            map_ax.set_xlabel("miles")
            map_ax.set_ylabel("miles")

            profile_ax.clear()
            widths_miles, elevs_ft = schematic_floor_profile_miles(table)
            profile_ax.fill_betweenx(
                elevs_ft, -widths_miles, widths_miles, color=ROCK_COLOR, alpha=0.5
            )
            below_water = elevs_ft <= elevation_ft
            water_elevs = np.append(elevs_ft[below_water], elevation_ft)
            water_widths = np.append(
                widths_miles[below_water],
                np.interp(elevation_ft, elevs_ft, widths_miles),
            )
            profile_ax.fill_betweenx(
                water_elevs, -water_widths, water_widths, color=WATER_COLOR
            )
            profile_ax.axhline(
                reservoir.full_pool_elevation_ft,
                color="#009E73",
                linewidth=1,
                linestyle="--",
                label="Full pool",
            )
            profile_ax.axhline(
                reservoir.minimum_power_pool_ft,
                color="#D55E00",
                linewidth=1,
                linestyle="--",
                label="Minimum power pool",
            )
            profile_ax.set_xlim(
                -max_radius_miles[reservoir.name] * 1.15,
                max_radius_miles[reservoir.name] * 1.15,
            )
            profile_ax.set_ylim(elevs_ft.min(), reservoir.full_pool_elevation_ft * 1.01)
            profile_ax.set_title(
                f"{reservoir.name} floor profile — elevation {elevation_ft:,.0f} ft"
            )
            profile_ax.set_xlabel("miles from centerline")
            profile_ax.set_ylabel("elevation (feet)")
            profile_ax.legend(loc="lower right", fontsize=8)

        fig.canvas.draw_idle()

    slider.on_changed(draw)
    draw(slider.val)